In [ ]:
# Imports and setup
import sys
from pathlib import Path
# Ensure the repository `src` folder is on sys.path so `from utils...` works when running cells
sys.path.append(str(Path('../../src').resolve()))

import pandas as pd
import ast
from pathlib import Path
from joblib import dump
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import numpy as np
import traceback, time
from sklearn.base import clone

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler


from utils.models import MODELS
from utils.eval_metrics import evaluate_model


In [9]:
# Paths
BASE_DIR = Path("../../data/out/best_models_ml")
MODELS_DIR = BASE_DIR / "models"
PLOTS_DIR = BASE_DIR / "plots"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Cargar Excel con mejores modelos
best_df = pd.read_excel(BASE_DIR / "in" / "best_models_ml.xlsx")
best_df.head()

,variant,model,status,best_params,R2,MAE,MSE,RMSE,MAPE(%),n_total,n_train,n_test,train_time_s,_variant_meta,best_neg_rmse
0,v1_original,ElasticNet,grid_search_done,"{'alpha': 0.1, 'l1_ratio': 0.5}",0.421227,211.656081,145440.836294,381.367062,33.558248,48192,38553,9639,4.855204,"{'name': 'v1_original', 'description': 'All va...",-162.720240
1,v1_original_lags,Lasso,trained,0,0.946790,46.409314,13377.826295,115.662554,11.331305,48168,38534,9634,2.160996,"{'name': 'v1_original_lags', 'description': 'A...",NaN
2,v2_with_calendar,ElasticNet,grid_search_done,"{'alpha': 0.1, 'l1_ratio': 0.5}",0.420918,211.814378,145518.430015,381.468780,33.559185,48192,38553,9639,4.401157,"{'name': 'v2_with_calendar', 'description': 'A...",-162.771849
3,v2_with_calendar_lags,Lasso,trained,0,0.946779,46.490941,13380.591491,115.674507,11.361092,48168,38534,9634,2.116155,"{'name': 'v2_with_calendar_lags', 'description...",NaN
4,v3_no_solar,ElasticNet,grid_search_done,"{'alpha': 0.1, 'l1_ratio': 0.3}",0.415520,212.463809,146874.899358,383.242612,33.850087,48192,38553,9639,3.974993,"{'name': 'v3_no_solar', 'description': 'All va...",-164.942648


In [10]:
#Functions

def parse_params(p):
    if isinstance(p, str):
        try:
            return ast.literal_eval(p)
        except Exception:
            return {}
    return p if isinstance(p, dict) else {}

def plot_predictions(y_test, y_pred, out_path, n_points=200):
    plt.figure(figsize=(12,5))
    plt.plot(y_test.values[:n_points], label="Real")
    plt.plot(y_pred[:n_points], label="Predicho")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()

def plot_residuals(y_test, y_pred, out_path, n_points=200):
    resid = y_test.values - y_pred
    plt.figure(figsize=(12,4))
    plt.plot(resid[:n_points], color="tab:red")
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()

def plot_scatter(y_test, y_pred, out_path, sample=5000):
    idx = np.arange(len(y_test))
    if len(idx) > sample:
        idx = np.random.default_rng(42).choice(idx, size=sample, replace=False)

    plt.figure(figsize=(5,5))
    plt.scatter(y_test.values[idx], y_pred[idx], s=6, alpha=0.5)

    # Línea de ajuste perfecto (y = x)
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)

    plt.xlabel("Real")
    plt.ylabel("Predicho")
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close()

def _prepare_variant_data(df, target_column= 'PRECIO', categorical_numeric=None):
    """Given a dataframe, prepare X/y and preprocessor based on default rules used above.

    Returns: preprocessor, X_train, X_test, y_train, y_test
    """
    if categorical_numeric is None:
        categorical_numeric = ["YEAR", "MONTH", "DAY", "HORA", "NIVEL_ENSO", "DIA_SEMANA", "FESTIVO"]

    numeric_features = [col for col in df.columns if col not in categorical_numeric and col != target_column and col != 'FECHA_HORA']
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), numeric_features),
        ('cat', 'passthrough', [c for c in categorical_numeric if c in df.columns])
    ])

    X = df.drop(columns=['FECHA_HORA', target_column], errors='ignore')
    y = df[target_column]

    train_size = int(len(df) * 0.8)
    X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
    y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

    return preprocessor, X_train, X_test, y_train, y_test


In [13]:
rows = []
categorical_numeric = ["YEAR", "MONTH", "DAY", "HORA", "NIVEL_ENSO", "DIA_SEMANA", "FESTIVO"]

for _, row in best_df.iterrows():
    variant = row["variant"]
    model_name = row["model"]
    base_estimator = MODELS.get(model_name)
    if base_estimator is None:
        print(f"Modelo {model_name} no está en MODELS, se omite.")
        continue

    def parse_params(p):
        if isinstance(p, str):
            try:
                return ast.literal_eval(p)
            except Exception:
                return {}
        return p if isinstance(p, dict) else {}

    best_params = parse_params(row.get("best_params", {}))

    # Path csv variant
    vmeta = ast.literal_eval(str(row["_variant_meta"]))
    csv_path = Path(vmeta["saved_variant_csv"])

    try:
        df = pd.read_csv(csv_path)
        preproc, X_train, X_test, y_train, y_test = _prepare_variant_data(
            df, target_column="PRECIO", categorical_numeric=categorical_numeric
        )

        # base + best_params
        base_params = base_estimator.get_params()
        merged = {**base_params, **best_params}

        # clone and set params
        estimator = clone(base_estimator).set_params(**merged)

        # Debug: XGBoost
        if model_name == "XGBoost":
            print(f"[DEBUG] Parámetros efectivos de XGBoost ({variant}):")
            for k, v in estimator.get_params().items():
                if k in ("random_state", "n_jobs", "n_estimators", "learning_rate", "max_depth"):
                    print(f"  {k}: {v}")

        # Build pipeline
        pipe = Pipeline([("preprocessor", preproc), ("model", estimator)])

        start = time.time()
        pipe.fit(X_train, y_train)
        elapsed = time.time() - start

        y_pred = pipe.predict(X_test)
        metrics = evaluate_model(pipe, X_test, y_test)
        metrics.update({
            "variant": variant,
            "model": model_name,
            "train_time_s": elapsed,
            "n_train": len(y_train),
            "n_test": len(y_test),
        })
        rows.append(metrics)

        # Save trained model
        dump(pipe, MODELS_DIR / f"{variant}_{model_name}.joblib")

        # Plot
        plot_predictions(y_test, y_pred, PLOTS_DIR / f"pred_{variant}_{model_name}.png")
        plot_residuals(y_test, y_pred, PLOTS_DIR / f"resid_{variant}_{model_name}.png")
        plot_scatter(y_test, y_pred, PLOTS_DIR / f"scatter_{variant}_{model_name}.png")

        print(f"OK {variant}-{model_name}: MAE={metrics.get('MAE'):.2f}")

    except Exception as e:
        print(f"ERROR {variant}-{model_name}: {e}")
        traceback.print_exc()


# Consolidado
if rows:
    df_out = pd.DataFrame(rows)
    df_out.to_excel(BASE_DIR / "best_models_metrics.xlsx", index=False)

OK v1_original-ElasticNet: MAE=211.66
OK v1_original_lags-Lasso: MAE=46.41
OK v2_with_calendar-ElasticNet: MAE=211.81
OK v2_with_calendar_lags-Lasso: MAE=46.49
OK v3_no_solar-ElasticNet: MAE=212.46
OK v3_no_solar_lags-Lasso: MAE=46.23


c:\Users\Camilo\anaconda3\envs\env_tf\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


OK v4_no_fuel_consumption-MLP Regressor: MAE=219.75
OK v4_no_fuel_consumption_lags-Lasso: MAE=46.49
[DEBUG] Parámetros efectivos de XGBoost (v5_no_fuel_and_cost):
  learning_rate: 0.04125491713916763
  max_depth: 3
  n_estimators: 456
  n_jobs: -1
  random_state: 42
OK v5_no_fuel_and_cost-XGBoost: MAE=185.58
OK v5_no_fuel_and_cost_lags-Lasso: MAE=46.49
OK v6_no_econ_fuel_cost-MLP Regressor: MAE=185.09
OK v6_no_econ_fuel_cost_lags-Lasso: MAE=46.49
OK v7_only_gen_enso-Linear Regression: MAE=236.75
OK v7_only_gen_enso_lags-Linear Regression: MAE=236.75
OK v8_only_gen_no_solar_enso-Linear Regression: MAE=236.75
[DEBUG] Parámetros efectivos de XGBoost (v8_only_gen_no_solar_enso_lags):
  learning_rate: 0.05080599114588148
  max_depth: 4
  n_estimators: 348
  n_jobs: -1
  random_state: 42
OK v8_only_gen_no_solar_enso_lags-XGBoost: MAE=234.89
OK v9_date_range_all-Decision Tree: MAE=100.63
OK v9_date_range_all_lags-Lasso: MAE=25.85
OK v10_date_range_no_solar-Decision Tree: MAE=121.07
OK v10_dat